# Lecture 1 — Text Preprocessing and Tokenization

This lecture combines **theory, mathematical examples, and practical Hugging Face implementations**.

The central objective: understand how raw text is transformed into token IDs that a Transformer can process.

## The complete pipeline

| Stage | Description |
|---|---|
| **Raw text** | `"Hello, how are you?"` |
| **Preprocessing** | Normalization, cleaning, etc. |
| **Tokenization** | Split text into subword tokens |
| **Token-to-ID mapping** | Convert tokens into integers |
| **Input IDs** | $\mathbf{x}\in\mathbb{N}^{L}$, ready for the embedding layer |

> **Note:** cells that download pretrained tokenizers (`from_pretrained`) need an internet connection.

In [ ]:
!pip install -q transformers tokenizers datasets sentencepiece torch

# 1. Text preprocessing and tokenization

## 1.1 What is text preprocessing?

Text preprocessing is the set of operations applied to raw text before feeding it into a machine learning pipeline.

Examples:

- Lowercasing
- Removing unnecessary whitespace
- Handling punctuation
- Unicode normalization
- Removing HTML tags when working with scraped text

In [ ]:
import re
import unicodedata

text = "   HELLO,    WORLD!   "

# Normalize Unicode representation
text = unicodedata.normalize("NFC", text)

# Remove leading and trailing whitespace
text = text.strip()

# Normalize consecutive whitespace
text = re.sub(r"\s+", " ", text)

print(text)

Output: `HELLO, WORLD!`

Notice that we did **not** lowercase the text. This is intentional: preprocessing depends on the model and task.

For example, lowercasing can destroy useful information:

- `Apple` (company)
- `apple` (fruit)

Modern tokenizers often perform their own normalization internally.

## 1.2 What is tokenization?

Tokenization is the process of dividing text into smaller units called **tokens**.

Suppose we have: `I love machine learning.`

Word-level tokenization:

In [ ]:
text = "I love machine learning."
tokens = text.split()
print(tokens)

Output: `['I', 'love', 'machine', 'learning.']`

But this approach has limitations:

- Punctuation is attached to words.
- Different forms of a word become separate vocabulary entries.
- New or unseen words create problems.
- Vocabulary size can become very large.

This motivates **subword tokenization**.

# 2. Character, word, and subword tokenization

There are three major approaches.

## 2.1 Character-level tokenization

Each character becomes an individual token.

In [ ]:
text = "hello"
tokens = list(text)
print(tokens)

**Advantages**

- Very small vocabulary.
- Almost no out-of-vocabulary problem.
- Can represent arbitrary words.

**Disadvantages**

- Very long sequences.
- The model must learn word-level patterns from character sequences.
- More computationally expensive for long texts.

Mathematically, for a sequence of $N$ characters:

$$T_{\text{char}} = N$$

## 2.2 Word-level tokenization

Each word becomes one token.

In [ ]:
text = "I love deep learning"
tokens = text.split()
print(tokens)

**Advantages**

- Shorter sequences.
- Words are preserved as meaningful units.

**Disadvantages**

- Large vocabulary.
- Unseen words create an out-of-vocabulary (OOV) problem.
- Morphological variations increase vocabulary size.

For example, `play`, `playing`, `played`, `player` — a word-level tokenizer may treat these as four completely different tokens.

## 2.3 Subword tokenization

Subword tokenization represents words using smaller units, which can be complete words or parts of words.

For example: `unbelievable → un + believe + able`

The exact segmentation depends on the tokenizer's vocabulary and algorithm.

**Why is it useful?** It provides a compromise:

- Smaller vocabulary than word-level tokenization.
- Shorter sequences than character-level tokenization.
- Better handling of unseen words.

A quick look at a real subword tokenizer (BERT's WordPiece) — requires internet:

In [ ]:
from transformers import AutoTokenizer

bert = AutoTokenizer.from_pretrained("bert-base-uncased")
for w in ["play", "playing", "unbelievable", "tokenization"]:
    print(f"{w:15s} ->", bert.tokenize(w))

# 3. Byte Pair Encoding (BPE) — in detail

This is the most important part of today's lecture.

## 3.1 The main idea

BPE is a subword tokenization algorithm that builds a vocabulary by **repeatedly merging the most frequent adjacent pair of tokens**.

Originally, BPE was a data compression algorithm. It was adapted for NLP to construct subword vocabularies.

## 3.2 Starting with a corpus

Suppose our training corpus contains:

| Word | Frequency |
| ---- | --------- |
| hug  | 10        |
| pug  | 5         |
| pun  | 12        |
| bun  | 4         |

Initially, we represent each word as a sequence of characters:

```
h u g : 10
p u g : 5
p u n : 12
b u n : 4
```

Our initial vocabulary is:

$$V_0 = \{h,u,g,p,n,b\}$$

The frequency of each word is retained because we want frequent character combinations to have a higher chance of being merged.

## 3.3 Step 1: Count adjacent pairs

```
hug → (h,u), (u,g)
pug → (p,u), (u,g)
pun → (p,u), (u,n)
bun → (b,u), (u,n)
```

| Pair   | Frequency |
| ------ | --------- |
| (h, u) | 10        |
| (u, g) | 15 (10 + 5) |
| (p, u) | 17 (5 + 12) |
| (u, n) | 16 (12 + 4) |
| (b, u) | 4         |

Therefore:

$$(p,u) = \arg\max_{(a,b)} f(a,b)$$

We merge `p` and `u` into `pu`.

## 3.4 Step 2: Update the corpus

```
h u g : 10
pu g : 5
pu n : 12
b u n : 4
```

Now $V_1 = V_0 \cup \{pu\}$. We repeat the process. The new pair frequencies are:

| Pair    | Frequency |
| ------- | --------- |
| (h, u)  | 10        |
| (u, g)  | 10        |
| (pu, g) | 5         |
| (pu, n) | 12        |
| (b, u)  | 4         |
| (u, n)  | 4         |

The most frequent pair is now `(pu, n)`. We merge it into `pun`.

## 3.5 Step 3: Repeat

$$V_{t+1}=V_t\cup\{a_t b_t\}$$

where

$$(a_t,b_t)= \arg\max_{(a,b)} f_t(a,b)$$

We stop when we reach the desired vocabulary size or number of merges.

> **Important distinction:** BPE learns **merge rules** during training. Those rules are then applied **in order** during tokenization.

## 3.6 Implement BPE yourself in Python

This implementation is useful for teaching because you can see exactly what happens at every iteration.

In [ ]:
from collections import Counter

# Corpus: word -> frequency
corpus = {
    "hug": 10,
    "pug": 5,
    "pun": 12,
    "bun": 4,
}

# Initialize each word as a sequence of characters
vocab = {tuple(word): frequency for word, frequency in corpus.items()}


def get_pair_frequencies(vocab):
    """Count adjacent pairs, weighted by word frequency."""
    pairs = Counter()
    for word, frequency in vocab.items():
        for i in range(len(word) - 1):
            pair = (word[i], word[i + 1])
            pairs[pair] += frequency
    return pairs


def merge_pair(pair, vocab):
    """Replace every occurrence of `pair` by the merged symbol."""
    new_vocab = {}
    for word, frequency in vocab.items():
        new_word = []
        i = 0
        while i < len(word):
            if i < len(word) - 1 and (word[i], word[i + 1]) == pair:
                new_word.append(word[i] + word[i + 1])
                i += 2
            else:
                new_word.append(word[i])
                i += 1
        key = tuple(new_word)
        new_vocab[key] = new_vocab.get(key, 0) + frequency
    return new_vocab


num_merges = 5
merges = []          # learned merge rules, in order
symbols = sorted({ch for word in vocab for ch in word})
print("Initial vocabulary:", symbols)

for step in range(1, num_merges + 1):
    pairs = get_pair_frequencies(vocab)
    if not pairs:
        break
    best_pair, best_freq = pairs.most_common(1)[0]
    vocab = merge_pair(best_pair, vocab)
    merges.append(best_pair)
    symbols.append(best_pair[0] + best_pair[1])

    print(f"\n--- Merge {step}: {best_pair} (frequency {best_freq}) ---")
    print("Pair frequencies:", dict(pairs))
    for word, freq in vocab.items():
        print("  ", " ".join(word), ":", freq)

print("\nLearned merges:", merges)
print("Final vocabulary:", symbols)

### Understanding the implementation

1. `Counter` counts how frequently each adjacent pair occurs.
2. `get_pair_frequencies()` calculates pair frequencies, taking word frequencies into account.
3. `merge_pair()` replaces every occurrence of the selected adjacent pair.
4. `most_common(1)` selects the pair with the highest frequency.
5. We repeat the operation for the specified number of merges.

Now let's **apply the learned merge rules, in order**, to tokenize a word (this is the second half of BPE: training vs. tokenization):

In [ ]:
def bpe_tokenize(word, merges):
    tokens = list(word)
    for a, b in merges:                      # apply rules in the order they were learned
        i = 0
        while i < len(tokens) - 1:
            if tokens[i] == a and tokens[i + 1] == b:
                tokens[i:i + 2] = [a + b]
            else:
                i += 1
    return tokens

for w in ["hug", "pun", "bug", "pugs"]:
    print(w, "->", bpe_tokenize(w, merges))

Notice that this is a **simplified teaching implementation**. Production tokenizers also handle special symbols, word boundaries, Unicode, byte representations, and more.

## 3.7 BPE limitations

- Greedy frequency-based merging does not necessarily produce linguistically meaningful units.
- The resulting vocabulary depends on the training corpus.
- Different languages and domains can produce different tokenization behavior.
- Standard character-based BPE can struggle with unseen characters.

This leads us to **byte-level BPE**, used by tokenizers such as GPT-2.

# 4. WordPiece and Unigram

You don't need to implement these from scratch today. Focus on their fundamental differences.

## 4.1 WordPiece

WordPiece is used by BERT. It also constructs a subword vocabulary, but its merge selection criterion differs from standard frequency-based BPE.

A simplified example: `playing → play + ##ing`

The `##` indicates that the token is a **continuation** of a word rather than its beginning.

The important distinction:

- **BPE** learns and stores merge rules.
- **WordPiece** stores the resulting vocabulary and tokenizes using a **longest-match-first** strategy.

For example, given the vocabulary below, WordPiece tries to find the longest valid subword at each position.

In [ ]:
vocabulary = {"play", "##ing", "##ed", "played"}

def wordpiece_tokenize(word, vocabulary):
    """Toy longest-match-first WordPiece tokenizer."""
    tokens, start = [], 0
    while start < len(word):
        end, piece = len(word), None
        while start < end:
            candidate = word[start:end]
            if start > 0:
                candidate = "##" + candidate
            if candidate in vocabulary:
                piece = candidate
                break
            end -= 1
        if piece is None:
            return ["[UNK]"]
        tokens.append(piece)
        start = end
    return tokens

for w in ["playing", "played", "play", "plays"]:
    print(w, "->", wordpiece_tokenize(w, vocabulary))

The actual WordPiece **training** algorithm uses a likelihood-based scoring criterion, rather than simply choosing the most frequent pair.

## 4.2 Unigram Language Model

Unigram takes a different approach. Instead of starting with individual characters and repeatedly merging them, it starts with a **large vocabulary of candidate subwords** and progressively **removes** less useful tokens.

It assigns probabilities to tokens: $P(t)$.

For a segmentation $X=(t_1,t_2,\ldots,t_k)$ the model approximates:

$$P(X)=\prod_{i=1}^{k}P(t_i)$$

It chooses a segmentation with high probability. Unigram is commonly used in **SentencePiece**.

## 4.3 Comparison

| Algorithm | Main mechanism                                    | Example use      |
| --------- | ------------------------------------------------- | ---------------- |
| BPE       | Merge frequent adjacent pairs                     | GPT-2, many LLMs |
| WordPiece | Vocabulary learning + longest-match segmentation  | BERT             |
| Unigram   | Probabilistic vocabulary pruning and segmentation | T5/SentencePiece |

*Note:* Model tokenizers have implementation-specific details. For example, modern LLM tokenizers often use byte-level BPE rather than the simplified character-level BPE introduced above.

# 5. Hugging Face Tokenizers

Now we move from theory to practice. We use two libraries:

- `transformers`: load and use pretrained tokenizers.
- `tokenizers`: build and train tokenizers from scratch efficiently.

## 5.1 Installation

```
pip install transformers tokenizers datasets
```
(Already done in the first cell.)

## 5.2 Load a pretrained tokenizer

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print(tokenizer)
print(type(tokenizer))
print(type(tokenizer.backend_tokenizer))

### Explanation

**`AutoTokenizer`** automatically selects the correct tokenizer implementation based on the model configuration. The argument `"bert-base-uncased"` identifies the pretrained model repository.

**`backend_tokenizer`** is the fast Rust-based tokenizer implementation, when a fast tokenizer is available. It handles normalization, pre-tokenization, subword segmentation, and token-to-ID conversion.

## 5.3 Tokenize a sentence

In [ ]:
text = "I love Natural Language Processing!"
tokens = tokenizer.tokenize(text)
print(tokens)

Possible output: `['i', 'love', 'natural', 'language', 'processing', '!']`

Why lowercase? Because `bert-base-uncased` is configured to normalize text to lowercase. Try it:

In [ ]:
text = "ChatGPT IS AMAZING!"
print("uncased:", tokenizer.tokenize(text))

# Compare with the cased model
tokenizer_cased = AutoTokenizer.from_pretrained("bert-base-cased")
print("cased:  ", tokenizer_cased.tokenize(text))

The cased model preserves case information.

## 5.4 Convert tokens into IDs

The tokenizer maintains a vocabulary

$$V=\{t_0,t_1,\ldots,t_{|V|-1}\}$$

and a mapping

$$f:V\rightarrow\{0,1,\ldots,|V|-1\}$$

Every token is assigned a unique integer ID.

In [ ]:
text = "I love machine learning"
tokens = tokenizer.tokenize(text)
token_ids = tokenizer.convert_tokens_to_ids(tokens)

print("Tokens:   ", tokens)
print("Token IDs:", token_ids)

The exact IDs depend on the pretrained vocabulary. You can also inspect the vocabulary:

In [ ]:
vocab = tokenizer.get_vocab()
print("Vocabulary size:", len(vocab))
print("ID of love:", vocab.get("love"))

Remember that a vocabulary is essentially a mapping between token strings and integer IDs.

## 5.5 Use `encode()`

Instead of performing tokenization and ID conversion separately, Hugging Face provides `encode()`. By default it adds special tokens according to the tokenizer configuration.

In [ ]:
text = "I love machine learning"
encoded = tokenizer.encode(text)
print(encoded)

For BERT, the resulting sequence typically looks like: `[CLS] i love machine learning [SEP]`

These special tokens have specific IDs:

In [ ]:
print(tokenizer.cls_token, tokenizer.cls_token_id)
print(tokenizer.sep_token, tokenizer.sep_token_id)
print(tokenizer.pad_token, tokenizer.pad_token_id)
print(tokenizer.unk_token, tokenizer.unk_token_id)
print(tokenizer.mask_token, tokenizer.mask_token_id)

**What are special tokens?**

- `[CLS]`: traditionally used as a classification representation.
- `[SEP]`: separates sequences or marks the end of a sequence.
- `[PAD]`: padding token.
- `[UNK]`: unknown token.
- `[MASK]`: used in masked language modeling.

Different model families use different special tokens and conventions.

## 5.6 Decode token IDs back into text

The reverse operation is called **decoding**.

In [ ]:
text = "I love machine learning"
encoded = tokenizer.encode(text)
decoded = tokenizer.decode(encoded)
print(decoded)

You can also inspect each stage — this is useful when debugging tokenization:

In [ ]:
tokens = tokenizer.convert_ids_to_tokens(encoded)

print("IDs:    ", encoded)
print("Tokens: ", tokens)
print("Decoded:", tokenizer.decode(encoded))

## 5.7 Understand the complete tokenizer output

A particularly important Hugging Face call is `tokenizer(...)`.

In [ ]:
text = "I love machine learning"
output = tokenizer(text)
print(output)

You will typically receive `input_ids`, `token_type_ids` and `attention_mask`. Let's understand each field.

**1. `input_ids`** — the integer IDs representing the tokens:

$$\mathbf{x}=(x_1,x_2,\ldots,x_L), \qquad x_i\in\{0,\ldots,|V|-1\}$$

**2. `attention_mask`** — a binary mask indicating which positions are actual tokens and which are padding:

$$m_i= \begin{cases} 1 & \text{real token}\\ 0 & \text{padding} \end{cases}$$

**3. `token_type_ids`** — used by models such as BERT to distinguish between two input sequences. For a sentence-pair task:

```
[CLS] Sentence A [SEP] Sentence B [SEP]
```

The first sequence can have segment ID 0 and the second segment ID 1. Not every architecture uses `token_type_ids`.

In [ ]:
# Sentence-pair example: look at token_type_ids
pair = tokenizer("I love machine learning", "Transformers are powerful")
print(tokenizer.convert_ids_to_tokens(pair["input_ids"]))
print(pair["token_type_ids"])

## 5.8 Padding and truncation

Essential when preparing batches for neural networks. Imagine two sentences with different numbers of tokens. Neural network batches generally require **rectangular tensors**:

$$X\in\mathbb{N}^{B\times L}$$

where $B$ is the batch size and $L$ the sequence length. Padding makes all sequences in the batch have the same length.

In [ ]:
texts = [
    "I love machine learning",
    "Transformers are powerful models",
]

batch = tokenizer(
    texts,
    padding=True,
    truncation=True,
    return_tensors="pt",
)

print(batch["input_ids"])
print(batch["attention_mask"])
print("Shape:", batch["input_ids"].shape)

**`padding=True`** adds `[PAD]` tokens to shorter sequences until all sequences have the same length. **`truncation=True`** cuts sequences that exceed the maximum length. For example:

In [ ]:
batch = tokenizer(
    texts,
    padding="max_length",
    truncation=True,
    max_length=10,
    return_tensors="pt",
)

print(batch["input_ids"].shape)   # torch.Size([2, 10])
print(batch["input_ids"])
print(batch["attention_mask"])

Now $X\in\mathbb{N}^{2\times 10}$.

### Important distinction

- `padding=True`: pad to the **longest sequence in the current batch**.
- `padding="max_length"`: pad to the specified `max_length`.
- `truncation=True`: truncate sequences that exceed the maximum.

## 5.9 Explore token offsets

Very useful in NLP applications such as named entity recognition. Offsets tell us which characters in the original text correspond to each token.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased", use_fast=True)

text = "I love machine learning."
encoding = tokenizer(text, return_offsets_mapping=True)

tokens = tokenizer.convert_ids_to_tokens(encoding["input_ids"])
offsets = encoding["offset_mapping"]

for token, offset in zip(tokens, offsets):
    print(token, offset)

An offset is represented as $(s_i, e_i)$ where $s_i$ is the starting character index and $e_i$ the ending character index (exclusive). For example `love → (2, 6)` means the token corresponds to `text[2:6]`:

In [ ]:
print(text[2:6])

Special tokens generally have `(0, 0)` offsets.

**Why is this useful?** Suppose a model predicts that a token corresponds to a person's name. We can use offsets to locate that prediction in the original sentence.

Note that offsets refer to the tokenizer's input string and depend on its normalization and alignment implementation.

# 6. Train your own tokenizer using Hugging Face

An excellent practical demonstration: students see how a tokenizer *learns* its vocabulary rather than simply downloading one. We'll train a **Byte-Level BPE** tokenizer.

## 6.1 Prepare a small corpus

In real training, you would use a much larger corpus.

In [ ]:
corpus_text = """I love machine learning.
Machine learning is transforming technology.
Deep learning is a subset of machine learning.
Transformers are powerful neural network architectures.
Natural language processing helps computers understand text.
Language models learn statistical patterns from text.
"""

with open("corpus.txt", "w", encoding="utf-8") as f:
    f.write(corpus_text)

print(open("corpus.txt").read())

## 6.2 Train a BPE tokenizer

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
from tokenizers.normalizers import NFKC

# 1. Initialize the BPE model
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))

# 2. Configure normalization
tokenizer.normalizer = NFKC()

# 3. Configure pre-tokenization
tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)

# 3b. Make decoding reversible
tokenizer.decoder = ByteLevelDecoder()

# 4. Configure the BPE trainer
trainer = BpeTrainer(
    vocab_size=500,
    min_frequency=2,
    special_tokens=["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"],
    initial_alphabet=ByteLevel.alphabet(),   # all 256 byte symbols
)

# 5. Train on the corpus
tokenizer.train(files=["corpus.txt"], trainer=trainer)

# 6. Save
tokenizer.save("my_tokenizer.json")

print("Vocabulary size:", tokenizer.get_vocab_size())

### Explanation of every component

| Component        | Responsibility                                                         |
| ---------------- | ---------------------------------------------------------------------- |
| `BPE()`          | Defines the tokenization model                                         |
| `NFKC()`         | Applies Unicode compatibility normalization                            |
| `ByteLevel()`    | Converts text into a byte-level representation during pre-tokenization |
| `BpeTrainer()`   | Learns the vocabulary and merge rules                                  |
| `vocab_size`     | Sets the target vocabulary size                                        |
| `min_frequency`  | Minimum frequency for eligible merges                                  |
| `special_tokens` | Reserves IDs for special tokens                                        |
| `train()`        | Learns the tokenizer from the corpus                                   |
| `save()`         | Saves the tokenizer configuration and learned model                    |

One subtlety: the byte-level pre-tokenizer operates on UTF-8 bytes and uses a reversible representation. This helps handle arbitrary Unicode input without requiring an unknown token for every unseen character.

Let's test the trained tokenizer, including on text that was **not** in the corpus:

In [ ]:
for s in ["machine learning is powerful", "Transformers are powerful!", "Bonjour مرحبا"]:
    enc = tokenizer.encode(s)
    print(s)
    print("  tokens:", enc.tokens)
    print("  ids:   ", enc.ids)
    print("  decode:", tokenizer.decode(enc.ids))

## 6.3 Load your trained tokenizer in Transformers

In [ ]:
from transformers import PreTrainedTokenizerFast

hf_tokenizer = PreTrainedTokenizerFast(
    tokenizer_file="my_tokenizer.json",
    unk_token="[UNK]",
    pad_token="[PAD]",
    cls_token="[CLS]",
    sep_token="[SEP]",
    mask_token="[MASK]",
)

text = "Transformers are powerful!"
print(hf_tokenizer.tokenize(text))
print(hf_tokenizer.encode(text))

Now we can use it with the familiar Hugging Face API.

> **Important:** Training a tokenizer does **not** train a language model. It only learns the vocabulary and tokenization rules. A language model must subsequently be trained or adapted using that tokenizer.

# 7. Compare tokenizers used by real LLMs

Different models can tokenize the same sentence differently.

In [ ]:
from transformers import AutoTokenizer

models = [
    "bert-base-uncased",
    "gpt2",
    "google/flan-t5-small",
]

text = "Tokenization is fundamental to language models."

for model_name in models:
    tok = AutoTokenizer.from_pretrained(model_name)
    tokens = tok.tokenize(text)
    ids = tok.convert_tokens_to_ids(tokens)

    print("\nModel:", model_name)
    print("Tokens:", tokens)
    print("IDs:", ids)
    print("Number of tokens:", len(tokens))

Explain that:

- **BERT** uses WordPiece.
- **GPT-2** uses byte-level BPE.
- **FLAN-T5** uses a SentencePiece-based Unigram tokenizer.

The same text can produce different token sequences and different token counts. This matters because **the tokenizer determines the sequence length presented to the model**.

# 8. From token IDs to embeddings

This connects today's lecture to the next one. A tokenizer produces **integer IDs, not semantic vectors**. Suppose:

$$\mathbf{x}=[101,2054,2003,102]$$

The embedding layer maps each ID to a dense vector. Let $V$ be the vocabulary size and $d$ the embedding dimension. The embedding matrix is

$$E\in\mathbb{R}^{V\times d}$$

For an input sequence $\mathbf{x}\in\mathbb{N}^{L}$ the embedding operation produces

$$H=E[\mathbf{x}], \qquad H\in\mathbb{R}^{L\times d}$$

For a batch of $B$ sequences $X\in\mathbb{N}^{B\times L}$, the result is

$$H\in\mathbb{R}^{B\times L\times d}$$

Demonstration with PyTorch:

In [ ]:
import torch
import torch.nn as nn

vocab_size = 10000
embedding_dim = 128

embedding = nn.Embedding(
    num_embeddings=vocab_size,
    embedding_dim=embedding_dim,
)

input_ids = torch.tensor([
    [10, 25, 43, 89],
    [12, 56, 78, 90],
])

output = embedding(input_ids)

print("Input shape:", input_ids.shape)
print("Embedding matrix:", embedding.weight.shape)
print("Output shape:", output.shape)

Expected output:

```
Input shape: torch.Size([2, 4])
Embedding matrix: torch.Size([10000, 128])
Output shape: torch.Size([2, 4, 128])
```

This is the transition from **discrete token IDs** to **continuous vector representations**.

# 9. Practical exercises

## Exercise 1 — Explore a pretrained tokenizer

Using `bert-base-uncased`:

- Tokenize five different sentences.
- Display tokens and token IDs.
- Decode the IDs.
- Compare the number of words with the number of tokens.

In [ ]:
# Exercise 1 — your code here
sentences = [
    # add five sentences
]

## Exercise 2 — Compare tokenizers

Compare BERT, GPT-2, and FLAN-T5.

- Which produces more tokens?
- How do they handle punctuation?
- What happens with an unfamiliar word?

In [ ]:
# Exercise 2 — your code here

## Exercise 3 — Train a tokenizer

- Prepare a small text corpus.
- Train a BPE tokenizer.
- Inspect its vocabulary.
- Test it on sentences that were not present in the corpus.

In [ ]:
# Exercise 3 — your code here

# 10. Recommended lecture timing

| Section                                  | Duration    |
| ---------------------------------------- | ----------- |
| Text preprocessing and tokenization      | 10 min      |
| Character, word and subword tokenization | 15 min      |
| BPE: theory and manual example           | 25 min      |
| WordPiece and Unigram                    | 10 min      |
| Hugging Face pretrained tokenizers       | 20 min      |
| Training a BPE tokenizer                 | 15 min      |
| Token IDs and embeddings                 | 10 min      |
| **Total**                                | **105 min** |

## Recommended official resources

- [Hugging Face Tokenizers documentation](https://huggingface.co/docs/tokenizers/index)
- [Transformers tokenizer API](https://huggingface.co/docs/transformers/main_classes/tokenizer)
- [Hugging Face NLP Course — Tokenizers](https://huggingface.co/learn/llm-course/chapter6/1)
- [GPT-2 tokenizer implementation](https://github.com/openai/gpt-2/blob/master/src/encoder.py)

## Key takeaways

Students should leave understanding three things:

1. Why subword tokenization is necessary.
2. How BPE learns a vocabulary through iterative merges.
3. How raw text becomes `input_ids`, which eventually become embeddings.

Don't spend too much time on the mathematical details of WordPiece and Unigram today. They need to understand the differences, not master three tokenizer-training algorithms in one lecture.